# 圖與框同步增強

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/11-augmentation/)

先執行環境格，再閱讀、執行下方完整實驗。各節互相獨立，不需要上一節的 runtime。

本節在 CPU 逐節執行並保存輸出；合成資料短訓練、L4 與部署紀錄見網站驗證頁。真實場景長訓練仍待安排。這些範例沒有預訓練權重，也不需下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.3.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 逐節範例預設 CPU；L4 訓練與部署的實測另見網站驗證頁。保留相同版本的 CPU/CUDA build。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
import torch


def horizontal_flip(image,boxes):
    width = image.shape[-1]
    result = boxes.clone()
    result[:,0] = width-boxes[:,2]
    result[:,2] = width-boxes[:,0]
    return image.flip(-1),result


def crop(image,boxes,left,top,width,height,min_visibility=.5):
    shifted = boxes-torch.tensor([left,top,left,top],dtype=boxes.dtype)
    clipped = shifted.clone()
    clipped[:,[0,2]] = clipped[:,[0,2]].clamp(0,width)
    clipped[:,[1,3]] = clipped[:,[1,3]].clamp(0,height)
    area = (boxes[:,2:]-boxes[:,:2]).prod(-1)
    visible = (clipped[:,2:]-clipped[:,:2]).clamp(min=0).prod(-1)
    keep = (visible > 0) & (visible/area >= min_visibility)
    return image[:,top:top+height,left:left+width],clipped[keep],keep


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    image = torch.zeros(3,64,64)
    image[0,12:28,8:24] = 1
    boxes = torch.tensor([[8.,12.,24.,28.]])
    labels = torch.tensor([0],dtype=torch.long)
    flipped, transformed = horizontal_flip(image,boxes)
    assert torch.equal(transformed,torch.tensor([[40.,12.,56.,28.]]))
    expected = torch.zeros_like(image)
    expected[0,12:28,40:56] = 1
    assert torch.equal(flipped,expected)
    twice, original_boxes = horizontal_flip(flipped,transformed)
    assert torch.equal(twice,image) and torch.equal(original_boxes,boxes)
    cropped, clipped, keep = crop(image,boxes,16,8,32,32,.5)
    assert torch.equal(clipped,torch.tensor([[0.,4.,8.,20.]]))
    cropped_labels = labels[keep]
    assert cropped_labels.tolist() == [0] and cropped_labels.dtype == torch.long
    assert keep.tolist() == [True]
    assert cropped[0,4:20,0:8].sum() == 128
    _,removed,keep_strict = crop(image,boxes,16,8,32,32,.6)
    strict_labels = labels[keep_strict]
    assert removed.shape == (0,4) and keep_strict.tolist() == [False]
    assert strict_labels.shape == (0,) and strict_labels.dtype == torch.long
    # A no-object image keeps empty boxes AND Long[0] labels through both operations.
    empty_image = torch.zeros_like(image)
    empty_labels = torch.empty(0,dtype=torch.long)
    flipped_empty,empty = horizontal_flip(empty_image,torch.empty(0,4))
    cropped_empty,empty,empty_keep = crop(flipped_empty,empty,16,8,32,32,.5)
    assert empty.shape == (0,4) and cropped_empty.sum() == 0
    assert empty_labels[empty_keep].shape == (0,) and empty_labels.dtype == torch.long
    print('flip box',transformed.tolist(),'double flip is identity')
    print('crop box',clipped.tolist(),'visible area 128/256=.5')
    print('labels after visibility .5/.6',cropped_labels.tolist(),strict_labels.tolist())
    print('no-object image: boxes [0,4], labels Long[0], pixels stay zero')


if __name__ == '__main__':
    main()


flip box [[40.0, 12.0, 56.0, 28.0]] double flip is identity
crop box [[0.0, 4.0, 8.0, 20.0]] visible area 128/256=.5
labels after visibility .5/.6 [0] []
no-object image: boxes [0,4], labels Long[0], pixels stay zero
